<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/headlineGenerationNLP_trainReadyPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing Notebook — News Headline Generation

## Final goal

Create model-ready files:

- `model_input`: cleaned article text, optionally with a task prefix such as `summarize:`
- `model_target`: cleaned headline/title
- train / validation / test CSV files



## 1. Import Libraries


- `GroupShuffleSplit` to reduce leakage across train/val/test when articles are very similar
- optional `langdetect` for language filtering
- optional Hugging Face tokenizer checks for token-length filtering

In [1]:
import os
import re
import html
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 160)

## Install Extra Packages in Colab

- `contractions` (for contraction expansion)
- `langdetect` (for language filtering)
- `transformers` (for tokenizer-based length checks)

Only True if this is the notebook first run
False if already installed after first run

In [2]:
INSTALL_OPTIONAL_PACKAGES = False

if INSTALL_OPTIONAL_PACKAGES:
    !pip -q install contractions langdetect transformers sentence-transformers
    print("Optional packages installed.")
else:
    print("Skipped package installation. Set INSTALL_OPTIONAL_PACKAGES=True if needed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.2 MB/s eta 0:00:00
Optional packages installed.


## 3. Mount Google Drive and Set Paths

Expected input columns:

- `article`
- `title`

Optional metadata columns:

- `date`
- `publication`
- `section`
- `author`
- `url`



In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running inside Google Colab, or Drive mount skipped.")

# Change this path to your actual sampled dataset file.
INPUT_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/sampled_articles_dataset.pkl"

OUTPUT_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"
if not os.path.exists("/content/drive/MyDrive"):
    OUTPUT_DIR = "/mnt/data/news_headline_model_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data


## 4. Load the Dataset

The sampled dataset already prepared, around 56k rows

At this stage, we are not doing linguistic analysis. We are only loading the base data that will eventually go to the model.

In [4]:
def load_dataframe(path):
    path = str(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    if path.endswith(".pkl") or path.endswith(".pickle"):
        return pd.read_pickle(path)
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError("Unsupported file type. Use .csv, .pkl, .pickle, or .parquet")

raw_df = load_dataframe(INPUT_PATH)
print("Loaded shape:", raw_df.shape)
display(raw_df.head(3))
print("Columns:")
print(raw_df.columns.tolist())

Loaded shape: (49860, 12)


,idx,article_idx,date,year,month,day,author,title,article,url,section,publication
0,-8290,-8290,2017-04-15 00:00:00,2017,4.0,15,Raymond Wong,It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In January, Sams...",https://mashable.com/2017/04/15/samsung-galaxy-s8-battery-no-explosion-fire/,None,Mashable
1,-16035,-16035,2017-11-06,2017,11.0,6,Andrea K. Scott,Five Centuries of Drawings at the Morgan,"The almost unbearably excellent show “Drawn to Greatness: Master Drawings from the Thaw Collection” begins with a love story. In 1954, the dealer Eugene Tha...",https://www.newyorker.com/magazine/2017/11/06/five-centuries-of-drawings-at-the-morgan,magazine,New Yorker
2,8290,8290,2019-08-09,2019,8.0,9,"Justin Hill, Opinion Contributor",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,"This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it hurts Texa...",https://thehill.com/blogs/congress-blog/politics/456873-retaliation-tweets-like-castros-to-discourage-political,None,The Hill


Columns:
['idx', 'article_idx', 'date', 'year', 'month', 'day', 'author', 'title', 'article', 'url', 'section', 'publication']


## 5. Basic Column Validation

The model needs an article as input and a headline/title as target.

In [5]:
required_cols = ["article", "title"]
missing = [col for col in required_cols if col not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Please rename your columns first.")

before_base = len(raw_df)
missing_mask = raw_df[["article", "title"]].isna().any(axis=1)
missing_rows = raw_df[missing_mask].copy()

model_base_df = raw_df.dropna(subset=["article", "title"]).copy()
model_base_df["article"] = model_base_df["article"].astype(str)
model_base_df["title"] = model_base_df["title"].astype(str)

after_base = len(model_base_df)
removed_base = before_base - after_base
removed_base_pct = (removed_base / before_base * 100) if before_base else 0.0

print("Why this step: article and title are mandatory because this is supervised headline generation.")
print("Rows before dropna:", before_base)
print("Rows after dropna :", after_base)
print(f"Removed rows      : {removed_base} ({removed_base_pct:.3f}%)")
if removed_base > 0:
    print("Sample removed rows with missing article/title:")
    show_cols = [c for c in ["article", "title", "url", "publication", "date"] if c in missing_rows.columns]
    display(missing_rows[show_cols].head(5))

Why this step: article and title are mandatory because this is supervised headline generation.
Rows before dropna: 49860
Rows after dropna : 49860
Removed rows      : 0 (0.000%)


## 6. Model-Safe Cleaning Functions

 **light transformer-level cleaning**, but with a bit stronger real-world cleanup for news data



For headline generation, we still preserve natural language as much as possible:

- Keep punctuation
- Keep numbers
- Keep capitalization
- Keep stopwords
- best not to stem
- best not to lemmatize
- best not to lowercase everything

What added here:

- handling for newswire prefixes (like `UPDATE 1-`, `CORRECTED`, location datelines)
- Removal of repetitive boilerplate lines in article bodies (`Reporting by ...`, `Editing by ...`, etc.)

Why?

Because these patterns are common in All The News dataset and can confuse the model, while not adding real semantic value for headline generation.

In [8]:
try:
    import contractions
    HAS_CONTRACTIONS = True
except Exception:
    HAS_CONTRACTIONS = False
    print("contractions library not installed. Using fallback contraction rules.")

try:
    from langdetect import detect_langs
    HAS_LANGDETECT = True
except Exception:
    HAS_LANGDETECT = False
    print("langdetect is not installed. Language filter will be skipped unless you install it.")

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False
    print("transformers is not installed. Tokenizer-length filter will be skipped unless you install it.")

fallback_contractions = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "i'm": "i am", "it's": "it is", "he's": "he is", "she's": "she is",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "you're": "you are", "we're": "we are", "they're": "they are",
    "i've": "i have", "we've": "we have", "they've": "they have",
    "i'll": "i will", "you'll": "you will", "we'll": "we will",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "hasn't": "has not", "haven't": "have not",
    "hadn't": "had not", "wouldn't": "would not", "shouldn't": "should not",
    "couldn't": "could not"
}

# Broadened patterns to catch prefixes with colons, dashes, or just trailing spaces
NEWSWIRE_PREFIX_PATTERNS = [
    r"^\s*(?:REFILE-)?(?:UPDATE|CORRECTED|WRAPUP|PREVIEW)(?:\s+\d+)?\s*[:\-]\s*",
    r"^\s*(?:UPDATE|PREVIEW|BRIEF)\b\s*[:\-]?\s*",
    r"^\s*\([A-Za-z]+\)\s*-\s*",
    r"^\s*[A-Z][A-Za-z\s\.'-]{2,40}\s*\([A-Za-z\.\s]+\)\s*-\s*",
    r"^\s*brief\b",
    r"^\s*update\s*\d*\s*[-:]",
    r"^\s*refile\s*-?\s*update\s*\d*\s*[-:]",
    r"^\s*wrapup\s*\d*\s*[-:]",
    r"^\s*press digest\b",
    r"^\s*factors to watch\b",
    r"^\s*preview\b",
    r"^\s*team by team analysis\b",
    r"^\s*pro rata podcast\b",
    r"^\s*cctv script\b",
    r"^\s*us stocks snapshot\b",
    r"^\s*rising:\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}\s*\|\s*TheHill\s*$"
]

BOILERPLATE_LINE_PATTERNS = [
    r"^\s*Reporting by\b",
    r"^\s*Additional reporting by\b",
    r"^\s*Editing by\b",
    r"^\s*Compiled by\b",
    r"^\s*For related stories\b",
    r"^\s*Click here\b",
    r"^\s*Follow us\b",
    r"^\s*Subscribe\b",
    r"^\s*\(This story has been corrected.*\)$",
    r"^\s*\(Reuters\)\s*-\s*$"
]

combined_boilerplate_line_pattern = re.compile(
    "|".join(BOILERPLATE_LINE_PATTERNS),
    flags=re.IGNORECASE
)

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", str(text))

def remove_html_entities_and_tags(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    return text

def normalize_quotes_dashes(text):
    replacements = {"“": '"', "”": '"', "‘": "'", "’": "'", "—": " - ", "–": " - ", "…": "..."}
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def remove_urls_and_emails(text):
    text = re.sub(r"http\S+|www\.\S+", " ", str(text))
    text = re.sub(r"\S+@\S+", " ", text)
    return text

def remove_newswire_prefixes(text):
    text = str(text)
    for pat in NEWSWIRE_PREFIX_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    return text

def strip_article_boilerplate(text):
    lines = re.split(r"\n+", str(text))
    cleaned_lines = [line.strip() for line in lines if line.strip() and not combined_boilerplate_line_pattern.search(line)]
    return "\n".join(cleaned_lines)

def fix_whitespace(text):
    text = str(text).replace(" ", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_title_punctuation(title):
    """Normalize spacing and repeated punctuation in headline targets only."""
    title = str(title).strip()
    title = re.sub(r"\s+", " ", title)
    title = re.sub(r"\s+([,.;:!?])", r"\1", title)
    title = re.sub(r"([,.;:!?]){2,}", r"\1", title)
    return title.strip()

def expand_contractions_text(text):
    text = str(text)
    if HAS_CONTRACTIONS:
        return contractions.fix(text)
    for c, e in fallback_contractions.items():
        text = re.sub(re.escape(c), e, text, flags=re.IGNORECASE)
    return text

def clean_for_transformer(text, expand_contractions_flag=False, is_title=False):
    text = normalize_unicode(text)
    text = remove_html_entities_and_tags(text)
    text = normalize_quotes_dashes(text)
    text = remove_urls_and_emails(text)
    text = remove_newswire_prefixes(text)

    # Prefix removal is essential for both body and title in news datasets
    text = remove_newswire_prefixes(text)

    if not is_title:
        text = strip_article_boilerplate(text)

    if expand_contractions_flag:
        text = expand_contractions_text(text)

    text = fix_whitespace(text)
    return text

## 7. Apply Transformer-Level Cleaning

This creates the two important clean text columns:

- `article_clean_transformer`
- `title_clean`


In [9]:
# True only if we want contractions like "don't" -> "do not"
# For news headline generation, False is usually safer because it preserves the original style
EXPAND_CONTRACTIONS = False

model_df = model_base_df.copy()

model_df["article_clean_transformer"] = model_df["article"].apply(
    lambda x: clean_for_transformer(x, expand_contractions_flag=EXPAND_CONTRACTIONS, is_title=False)
)
model_df["title_clean"] = model_df["title"].apply(
    lambda x: clean_for_transformer(x, expand_contractions_flag=EXPAND_CONTRACTIONS, is_title=True)
)

print("After cleaning:", model_df.shape)
display(model_df[["article", "article_clean_transformer", "title", "title_clean"]].head(3))


# Preprocessing audit utilities

INITIAL_ROWS = len(model_df)
preprocess_audit = []


def log_filter_step(step_name, reason, before_df, after_df, show_cols=None, sample_n=5):
    """Logging what was removed in this step + show a small sample for justification."""
    before_count = len(before_df)
    after_count = len(after_df)
    removed_count = before_count - after_count

    removed_idx = before_df.index.difference(after_df.index)
    removed_df = before_df.loc[removed_idx].copy()

    removed_pct_step = (removed_count / before_count * 100) if before_count else 0.0
    cumulative_removed = INITIAL_ROWS - after_count
    cumulative_removed_pct = (cumulative_removed / INITIAL_ROWS * 100) if INITIAL_ROWS else 0.0

    preprocess_audit.append({
        "step": step_name,
        "reason": reason,
        "before_rows": before_count,
        "after_rows": after_count,
        "removed_rows": removed_count,
        "removed_pct_step": round(removed_pct_step, 3),
        "cumulative_removed": cumulative_removed,
        "cumulative_removed_pct": round(cumulative_removed_pct, 3),
    })

    print("=" * 90)
    print(f"Step: {step_name}")
    print(f"Why this step: {reason}")
    print(f"Rows before: {before_count}")
    print(f"Rows after : {after_count}")
    print(f"Removed    : {removed_count} ({removed_pct_step:.3f}% of this step input)")
    print(f"Cumulative removed from start: {cumulative_removed} ({cumulative_removed_pct:.3f}%)")

    if removed_count > 0:
        print(f"Sample removed rows (up to {sample_n}):")
        if show_cols is None:
            show_cols = [c for c in ["title_clean", "article_clean_transformer"] if c in removed_df.columns]
        else:
            # title+article visibility when possible so outputs are easier to show
            show_cols = [c for c in show_cols if c in removed_df.columns]
            if "title_clean" in removed_df.columns and "title_clean" not in show_cols:
                show_cols.append("title_clean")
            if "article_clean_transformer" in removed_df.columns and "article_clean_transformer" not in show_cols:
                show_cols.append("article_clean_transformer")

        if show_cols:
            sample_view = removed_df[show_cols].head(sample_n).copy()
            # short previews so long article text is still readable in table form
            if "article_clean_transformer" in sample_view.columns:
                sample_view["article_preview"] = sample_view["article_clean_transformer"].astype(str).str.slice(0, 220)
            if "title_clean" in sample_view.columns:
                sample_view["title_preview"] = sample_view["title_clean"].astype(str).str.slice(0, 120)
            display(sample_view)
        else:
            display(removed_df.head(sample_n))
    else:
        print("No rows removed in this step.")


def log_skipped_step(step_name, reason, current_df):
    """Log skipped optional steps so the audit table is complete."""
    current_count = len(current_df)
    cumulative_removed = INITIAL_ROWS - current_count
    cumulative_removed_pct = (cumulative_removed / INITIAL_ROWS * 100) if INITIAL_ROWS else 0.0

    preprocess_audit.append({
        "step": step_name,
        "reason": reason,
        "before_rows": current_count,
        "after_rows": current_count,
        "removed_rows": 0,
        "removed_pct_step": 0.0,
        "cumulative_removed": cumulative_removed,
        "cumulative_removed_pct": round(cumulative_removed_pct, 3),
    })

    print("=" * 90)
    print(f"Step: {step_name}")
    print(f"Why this step: {reason}")
    print("Step status: skipped (no rows removed).")

After cleaning: (49860, 14)


,article,article_clean_transformer,title,title_clean
0,"Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In January, Sams...","Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In January, Samsu...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife
1,"The almost unbearably excellent show “Drawn to Greatness: Master Drawings from the Thaw Collection” begins with a love story. In 1954, the dealer Eugene Tha...","The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer Eugene Tha...",Five Centuries of Drawings at the Morgan,Five Centuries of Drawings at the Morgan
2,"This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it hurts Texa...","This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it hurts Texa...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,Retaliation tweets like Castro's to discourage political involvement must stop | TheHill


## 8. Remove Empty or Very Short Text After Cleaning

This is a safety check after cleaning.

Some rows may become empty or too short after URL/HTML/noise removal.

In [10]:
before_df = model_df.copy()

model_df = model_df[
    (model_df["article_clean_transformer"].str.len() > 100) &
    (model_df["title_clean"].str.len() > 5)
].copy()

log_filter_step(
    step_name="Safety char-length filter",
    reason="Remove rows that become too short after cleaning; very short body/title pairs are usually weak supervision.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer"]
)

Step: Safety char-length filter
Why this step: Remove rows that become too short after cleaning; very short body/title pairs are usually weak supervision.
Rows before: 49860
Rows after : 49854
Removed    : 6 (0.012% of this step input)
Cumulative removed from start: 6 (0.012%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
4941,,"Opinion by: Krystal Ball President Trump's approval ticked up to 49 percent - its highest mark this year, according to a new Hill-HarrisX survey released on...","Opinion by: Krystal Ball President Trump's approval ticked up to 49 percent - its highest mark this year, according to a new Hill-HarrisX survey released on...",
8634,,"Republican strategist Marissa Martinez predicted Monday that former Vice President Joe Biden will win the Democratic nomination, citing his fundraising prow...","Republican strategist Marissa Martinez predicted Monday that former Vice President Joe Biden will win the Democratic nomination, citing his fundraising prow...",
18797,How Trump's immigration crackdown could slam the U.S. economy,,,How Trump's immigration crackdown could slam the U.S. economy
24895,Gloria Estefan Gets Kennedy Center Honors,Brought to you by the editors of People en Español.,Brought to you by the editors of People en Español.,Gloria Estefan Gets Kennedy Center Honors
34066,,House Majority Whip James Clyburn (D-S.C.) touted Thursday a new infrastructure plan being put forth by Democrats. A spokesperson for Heritage Action for Am...,House Majority Whip James Clyburn (D-S.C.) touted Thursday a new infrastructure plan being put forth by Democrats. A spokesperson for Heritage Action for Am...,


## 9. Remove Duplicate Articles and Duplicate Input-Target Pairs

This prevents data leakage and repeated examples

The most important duplicate to remove is the duplicated `article_clean_transformer`, because the same article may appear with repeated or slightly different titles

In [11]:
before_df = model_df.copy()

# Showing clear duplicates before dropping.
dup_mask_article = before_df.duplicated(subset=["article_clean_transformer"], keep=False)
dup_evidence = before_df[dup_mask_article].copy()
if len(dup_evidence) > 0:
    dup_evidence["duplicate_group_size"] = dup_evidence.groupby("article_clean_transformer")["article_clean_transformer"].transform("size")
    print("Duplicate evidence before exact dedup (same article text appears multiple times):")
    display(
        dup_evidence[["title_clean", "article_clean_transformer", "duplicate_group_size"]]
        .sort_values("duplicate_group_size", ascending=False)
        .head(12)
    )
else:
    print("No exact article duplicates found before this step.")

model_df = model_df.drop_duplicates(subset=["article_clean_transformer"], keep="first").copy()
model_df = model_df.drop_duplicates(subset=["article_clean_transformer", "title_clean"], keep="first").copy()

log_filter_step(
    step_name="Exact duplicate removal",
    reason="Remove repeated article pairs to reduce memorization and data leakage across splits.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer", "duplicate_group_size"]
)

Duplicate evidence before exact dedup (same article text appears multiple times):


,title_clean,article_clean_transformer,duplicate_group_size
3815,Nashville Stars Lennon & Maisy Stella Talk Binging Riverdale & the Songs They Can't Get Out of Their Head!,"It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...",4
17250,RHOC's Tamra Judge Reveals Why She Googled 'Food Poisoning',"It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...",4
13241,Sanaa Lathan Reveals the Last Song That Was Stuck in Her Head,"It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...",4
42611,Actress Lake Bell Reveals Her Last Date Night: 'I'm Still Mourning',"It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...",4
4341,Slovenia PM: chance of autumn Brexit deal if Britain backs it,"SALZBURG, Austria (Reuters) - Slovenia Prime Minister Marjan Sarec said on Wednesday he believed a deal with Britain over its withdrawal from the European U...",2
646,Ex-coach of Yale women's soccer pleads guilty in admissions scandal,The former head coach of women's soccer at Yale University admitted on Thursday that he accepted bribes to help parents get their children into the Ivy Leag...,2
10180,Syrian SDF says not informed of any U.S. withdrawal plan,The U.S.-backed Syrian Democratic Forces (SDF) said on Friday it had not been informed of any plan to withdraw U.S. forces operating in Syria as part of the...,2
4997,A nation built together | TheHill,"The views expressed by authors are their own and not the views of The Hill. View the discussion thread. The Hill 1625 K Street, NW Suite 900 Washington DC 2...",2
11442,U.S. withdraws from U.N. Human Rights Council: U.S. Ambassador Haley,"The United States withdrew from the U.N. Human Rights Council on Tuesday after no other countries ""had the courage to join our fight"" to reform the ""hypocri...",2
4739,Article on UK getting post-Brexit financial services deal is incorrect - EU official,A report by the Times newspaper that Britain and the European Union had reached a tentative agreement on all aspects of a future partnership on services aft...,2


Step: Exact duplicate removal
Why this step: Remove repeated article pairs to reduce memorization and data leakage across splits.
Rows before: 49854
Rows after : 49840
Removed    : 14 (0.028% of this step input)
Cumulative removed from start: 20 (0.040%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
13241,Sanaa Lathan Reveals the Last Song That Was Stuck in Her Head,"It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...","It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...",Sanaa Lathan Reveals the Last Song That Was Stuck in Her Head
13842,Doctors aren't selling out for sandwiches | TheHill,"The views expressed by authors are their own and not the views of The Hill. View the discussion thread. The Hill 1625 K Street, NW Suite 900 Washington DC 2...","The views expressed by authors are their own and not the views of The Hill. View the discussion thread. The Hill 1625 K Street, NW Suite 900 Washington DC 2...",Doctors aren't selling out for sandwiches | TheHill
17250,RHOC's Tamra Judge Reveals Why She Googled 'Food Poisoning',"It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...","It's One Last Thing Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebroadcast at...",RHOC's Tamra Judge Reveals Why She Googled 'Food Poisoning'
17916,U.S. withdraws from U.N. Human Rights Council: U.S. Ambassador Haley,"The United States withdrew from the U.N. Human Rights Council on Tuesday after no other countries ""had the courage to join our fight"" to reform the ""hypocri...","The United States withdrew from the U.N. Human Rights Council on Tuesday after no other countries ""had the courage to join our fight"" to reform the ""hypocri...",U.S. withdraws from U.N. Human Rights Council: U.S. Ambassador Haley
22561,Article on UK getting post-Brexit financial services deal is incorrect: EU official,A report by the Times newspaper that Britain and the European Union had reached a tentative agreement on all aspects of a future partnership on services aft...,A report by the Times newspaper that Britain and the European Union had reached a tentative agreement on all aspects of a future partnership on services aft...,Article on UK getting post-Brexit financial services deal is incorrect: EU official


## 10. Extra Dedup Step (Loose Near-Duplicate Key)

In news datasets, removing only exact duplicates is usually not enough

The same story can appear more than once with very small edits, like extra spaces, punctuation differences, or short wire-service updates. Because of this,a *loose* dedup key was added

The idea normalizing the article text strongly (lowercase, remove punctuation/noise, fix spaces) and use the first part of the text to build a key. If two rows get the same loose key, they are treated as near-duplicates, and only one of them is kept

This helps reduce repetition and possible leakage across splits, so the model learns from more diverse examples

This is still not semantic deduplication, but it is a practical and fast method that works well for Colab-sized preprocessing

In [12]:
def build_loose_dedup_key(text, max_chars=700):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]

before_df = model_df.copy()
before_df["dedup_key_loose"] = before_df["article_clean_transformer"].apply(build_loose_dedup_key)

# Show evidence for loose near-duplicate groups before dropping.
loose_dup_mask = before_df.duplicated(subset=["dedup_key_loose"], keep=False)
loose_evidence = before_df[loose_dup_mask].copy()
if len(loose_evidence) > 0:
    loose_evidence["loose_group_size"] = loose_evidence.groupby("dedup_key_loose")["dedup_key_loose"].transform("size")
    print("Loose near-duplicate evidence before dedup (same normalized key appears multiple times):")
    display(
        loose_evidence[["title_clean", "article_clean_transformer", "dedup_key_loose", "loose_group_size"]]
        .sort_values("loose_group_size", ascending=False)
        .head(12)
    )
else:
    print("No loose near-duplicate groups found before this step.")

model_df["dedup_key_loose"] = model_df["article_clean_transformer"].apply(build_loose_dedup_key)
model_df = model_df.drop_duplicates(subset=["dedup_key_loose"], keep="first").copy()

log_filter_step(
    step_name="Loose near-duplicate removal",
    reason="Catch practical near-duplicates with tiny edits so the model sees more diverse examples.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer", "dedup_key_loose", "loose_group_size"]
)

Loose near-duplicate evidence before dedup (same normalized key appears multiple times):


,title_clean,article_clean_transformer,dedup_key_loose,loose_group_size
653,"Oil rises on OPEC-led supply cuts, report of falling US crude inventories","SINGAPORE, Feb 27 (Reuters) - Oil prices rose on Wednesday after a report of declining crude inventories in the country and as producer club OPEC seemed to ...",singapore feb 27 reuters oil prices rose on wednesday after a report of declining crude inventories in the country and as producer club opec seemed to stick...,2
2252,"Exxon shareholders reject splitting CEO, board chair roles",May 29 (Reuters) - Exxon Mobil Corp shareholders on Wednesday rejected a proposal that would have split the roles of the chief executive and board chairman ...,may 29 reuters exxon mobil corp shareholders on wednesday rejected a proposal that would have split the roles of the chief executive and board chairman at t...,2
7476,"Oil rises on OPEC-led supply cuts, report of falling US crude inventories","SINGAPORE, Feb 27 (Reuters) - Oil prices rose on Wednesday after a report of declining crude inventories in the country and as producer club OPEC seemed to ...",singapore feb 27 reuters oil prices rose on wednesday after a report of declining crude inventories in the country and as producer club opec seemed to stick...,2
9065,Water protests in tech hub expose urban India's growing pains,"Oracle employees were at work on Monday when protesters entered their nine-storey building in India's technology hub, Bengaluru, and asked them to leave in ...",oracle employees were at work on monday when protesters entered their nine storey building in india s technology hub bengaluru and asked them to leave in su...,2
14232,UBS loses role in bond deal for Chinese firm on outcry over pig comment,"UBS has lost a lead role on a U.S. dollar bond deal for state-backed China Railway Construction Corp, just days after a Chinese outcry over a senior UBS eco...",ubs has lost a lead role on a u s dollar bond deal for state backed china railway construction corp just days after a chinese outcry over a senior ubs econo...,2
19618,Water protests in tech hub expose urban India's growing pains,"Oracle employees were at work on Monday when protesters entered their nine-storey building in India's technology hub, Bengaluru, and asked them to leave in ...",oracle employees were at work on monday when protesters entered their nine storey building in india s technology hub bengaluru and asked them to leave in su...,2
21184,Sephora cuts ties with TV star's daughter after college cheating scam,LVMH's Sephora beauty chain ended its partnership with Olivia Jade following a massive college cheating scandal involving her celebrity parents who were cha...,lvmh s sephora beauty chain ended its partnership with olivia jade following a massive college cheating scandal involving her celebrity parents who were cha...,2
24393,UBS loses a bond deal in China after economist used 'Chinese pig' term,"UBS has lost a lead role on a U.S. dollar bond deal for state-backed China Railway Construction Corp, just days after a Chinese outcry over a senior UBS eco...",ubs has lost a lead role on a u s dollar bond deal for state backed china railway construction corp just days after a chinese outcry over a senior ubs econo...,2
31687,Holiday Inn-owner IHG takes hit from Hong Kong protests,"Feb 18 (Reuters) - Holiday Inn-owner InterContinental Hotels Group reported a slight dip in revenue per room in 2019, hurt by a fall in bookings in Hong Kon...",feb 18 reuters holiday inn owner intercontinental hotels group reported a slight dip in revenue per room in 2019 hurt by a fall in bookings in hong kong due...,2
33040,Holiday Inn-owner IHG takes hit from Hong Kong protests,"Feb 18 (Reuters) - Holiday Inn-owner InterContinental Hotels Group reported a slight dip in revenue per room in 2019, hurt by a fall in bookings in Hong Kon...",feb 18 reuters holiday inn owner intercontinental hotels group reported a slight dip in revenue per room in 2019 hurt by a fall in bookin

Step: Loose near-duplicate removal
Why this step: Catch practical near-duplicates with tiny edits so the model sees more diverse examples.
Rows before: 49840
Rows after : 49833
Removed    : 7 (0.014% of this step input)
Cumulative removed from start: 27 (0.054%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,dedup_key_loose,article_preview,title_preview
7476,"Oil rises on OPEC-led supply cuts, report of falling US crude inventories","SINGAPORE, Feb 27 (Reuters) - Oil prices rose on Wednesday after a report of declining crude inventories in the country and as producer club OPEC seemed to ...",singapore feb 27 reuters oil prices rose on wednesday after a report of declining crude inventories in the country and as producer club opec seemed to stick...,"SINGAPORE, Feb 27 (Reuters) - Oil prices rose on Wednesday after a report of declining crude inventories in the country and as producer club OPEC seemed to ...","Oil rises on OPEC-led supply cuts, report of falling US crude inventories"
19618,Water protests in tech hub expose urban India's growing pains,"Oracle employees were at work on Monday when protesters entered their nine-storey building in India's technology hub, Bengaluru, and asked them to leave in ...",oracle employees were at work on monday when protesters entered their nine storey building in india s technology hub bengaluru and asked them to leave in su...,"Oracle employees were at work on Monday when protesters entered their nine-storey building in India's technology hub, Bengaluru, and asked them to leave in ...",Water protests in tech hub expose urban India's growing pains
24393,UBS loses a bond deal in China after economist used 'Chinese pig' term,"UBS has lost a lead role on a U.S. dollar bond deal for state-backed China Railway Construction Corp, just days after a Chinese outcry over a senior UBS eco...",ubs has lost a lead role on a u s dollar bond deal for state backed china railway construction corp just days after a chinese outcry over a senior ubs econo...,"UBS has lost a lead role on a U.S. dollar bond deal for state-backed China Railway Construction Corp, just days after a Chinese outcry over a senior UBS eco...",UBS loses a bond deal in China after economist used 'Chinese pig' term
33040,Holiday Inn-owner IHG takes hit from Hong Kong protests,"Feb 18 (Reuters) - Holiday Inn-owner InterContinental Hotels Group reported a slight dip in revenue per room in 2019, hurt by a fall in bookings in Hong Kon...",feb 18 reuters holiday inn owner intercontinental hotels group reported a slight dip in revenue per room in 2019 hurt by a fall in bookings in hong kong due...,"Feb 18 (Reuters) - Holiday Inn-owner InterContinental Hotels Group reported a slight dip in revenue per room in 2019, hurt by a fall in bookings in Hong Kon...",Holiday Inn-owner IHG takes hit from Hong Kong protests
35664,"Exxon shareholders reject splitting CEO, board chair roles",May 29 (Reuters) - Exxon Mobil Corp shareholders on Wednesday rejected a proposal that would have split the roles of the chief executive and board chairman ...,may 29 reuters exxon mobil corp shareholders on wednesday rejected a proposal that would have split the roles of the chief executive and board chairman at t...,May 29 (Reuters) - Exxon Mobil Corp shareholders on Wednesday rejected a proposal that would have split the roles of the chief executive and board chairman ...,"Exxon shareholders reject splitting CEO, board chair roles"


### 10.1 Embedding-Based Near-Duplicate Removal

This catches articles that are semantically very similar even when they are not exact or loose text duplicates.

It is optional. Turn it on after the normal pipeline runs successfully.

In [14]:
USE_EMBEDDING_NEAR_DUPLICATE_FILTER = True
EMBEDDING_NEAR_DUP_THRESHOLD = 0.94
EMBEDDING_DEDUP_MODEL_NAME = "all-MiniLM-L6-v2"

if USE_EMBEDDING_NEAR_DUPLICATE_FILTER:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.neighbors import NearestNeighbors
        HAS_EMBEDDING_DEDUP = True
    except Exception:
        HAS_EMBEDDING_DEDUP = False
        print("sentence-transformers or sklearn NearestNeighbors missing. Embedding dedup will be skipped.")

    if HAS_EMBEDDING_DEDUP:
        print("Building embeddings for near-duplicate detection...")
        emb_model = SentenceTransformer(EMBEDDING_DEDUP_MODEL_NAME)

        # Use first 1500 chars because lead text usually identifies the news story well.
        dedup_texts = model_df["article_clean_transformer"].astype(str).str[:1500].tolist()
        embeddings = emb_model.encode(
            dedup_texts,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=64
        )

        # Cosine similarity threshold becomes cosine distance radius = 1 - similarity.
        radius = 1 - EMBEDDING_NEAR_DUP_THRESHOLD
        nn = NearestNeighbors(metric="cosine", radius=radius, algorithm="brute")
        nn.fit(embeddings)
        neighbors = nn.radius_neighbors(embeddings, return_distance=False)

        keep_positions = []
        removed_positions = set()

        for i, neigh in enumerate(neighbors):
            if i in removed_positions:
                continue
            keep_positions.append(i)
            for j in neigh:
                if j != i:
                    removed_positions.add(int(j))

        before_df = model_df.copy()
        model_df = model_df.iloc[keep_positions].copy()

        log_filter_step(
            step_name="Embedding-based near-duplicate removal",
            reason="Remove semantically similar duplicate stories that text-based dedup may miss.",
            before_df=before_df,
            after_df=model_df,
            show_cols=["title_clean", "article_clean_transformer"]
        )
else:
    log_skipped_step(
        step_name="Embedding-based near-duplicate removal",
        reason="Skipped by default because it can be slow; set USE_EMBEDDING_NEAR_DUPLICATE_FILTER=True to enable it.",
        current_df=model_df
    )


Building embeddings for near-duplicate detection...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/779 [00:00<?, ?it/s]

Step: Embedding-based near-duplicate removal
Why this step: Remove semantically similar duplicate stories that text-based dedup may miss.
Rows before: 49833
Rows after : 49728
Removed    : 105 (0.211% of this step input)
Cumulative removed from start: 132 (0.265%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
4189,FOREX-Dollar sturdy after biggest quarterly rise since June 2018,"* Graphic: World FX rates in 2019 tmsnrt.rs/2egbfVh By Saikat Chatterjee LONDON, Oct 1 (Reuters) - The dollar rose to a 29-month high on Tuesday as renewed ...","* Graphic: World FX rates in 2019 tmsnrt.rs/2egbfVh By Saikat Chatterjee LONDON, Oct 1 (Reuters) - The dollar rose to a 29-month high on Tuesday as renewed ...",FOREX-Dollar sturdy after biggest quarterly rise since June 2018
4907,Find Out What Show Shawn Johnson East Is Obsessed With!,"It's today's ""One Last Thing"" Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebr...","It's today's ""One Last Thing"" Come back every day at 8:30 a.m. EST to watch People Now streaming live from Time Inc. headquarters in New York City, and rebr...",Find Out What Show Shawn Johnson East Is Obsessed With!
8357,Bud Light debuts bigger nutrition labels,"Beer drinkers can't claim blissful ignorance for much longer. Starting next month, packages of Bud Light will have prominent labels showing the beer's calor...","Beer drinkers can't claim blissful ignorance for much longer. Starting next month, packages of Bud Light will have prominent labels showing the beer's calor...",Bud Light debuts bigger nutrition labels
9253,"Axios America: Dems win in OK, Bridgegate Kool-Aid, and monkey business","Welcome to today's Axios America, the latest on the day's top news from around the country, brought to you by Axios' Alayna Treene and Shannon Vavra. Check ...","Welcome to today's Axios America, the latest on the day's top news from around the country, brought to you by Axios' Alayna Treene and Shannon Vavra. Check ...","Axios America: Dems win in OK, Bridgegate Kool-Aid, and monkey business"
10004,"Today's Trump Top 5: Why Israel is ""boiling mad""","Welcome to today's Trump Top 5, brought to you by Axios for Apple News. For more, check out our news STREAM here. THANKS for reading! If you want the news t...","Welcome to today's Trump Top 5, brought to you by Axios for Apple News. For more, check out our news STREAM here. THANKS for reading! If you want the news t...","Today's Trump Top 5: Why Israel is ""boiling mad"""


## 11. Word Count Checks for Model Filtering

This part filters the actual data that will go to the model.

rules for a headline generation task:

- Article/input should not be too short
- Headline/target should not be too short
- Headline/target should not be too long

In [15]:
def simple_word_count(text):
    return len(str(text).split())

model_df["input_word_count"] = model_df["article_clean_transformer"].apply(simple_word_count)
model_df["target_word_count"] = model_df["title_clean"].apply(simple_word_count)

print("Input/article word counts:")
display(model_df["input_word_count"].describe())

print("Target/headline word counts:")
display(model_df["target_word_count"].describe())

Input/article word counts:


,input_word_count
count,49728.000000
mean,413.158221
std,239.299622
min,45.000000
25%,223.000000
50%,375.000000
75%,565.000000
max,1029.000000


Target/headline word counts:


,target_word_count
count,49728.000000
mean,10.133506
std,2.579711
min,3.000000
25%,8.000000
50%,10.000000
75%,12.000000
max,22.000000


## 12. Final Length Filtering (Word + tokenizer-level policy)

original word-based thresholds is kept because they are simple and stable

an optional tokenizer-based filter is added, because transformer models care about token count, not just word count

 Why this step:
 - Word limits are simple and good for first cleanup.
 - So we keep word filtering, then apply model-aware token handling:
 - truncate long inputs to model max length
 - drop only extreme outliers
 - keep realistic target token lengths

In [16]:
MIN_INPUT_WORDS = 80
MIN_TARGET_WORDS = 3
MAX_TARGET_WORDS = 25

before_df = model_df.copy()

model_df = model_df[
    (model_df["input_word_count"] >= MIN_INPUT_WORDS) &
    (model_df["target_word_count"] >= MIN_TARGET_WORDS) &
    (model_df["target_word_count"] <= MAX_TARGET_WORDS)
].copy()

log_filter_step(
    step_name="Word-length filter",
    reason="Keep rows where article has enough context and headline length looks realistic for training.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "input_word_count", "target_word_count"]
)


# Model-agnostic tokenizer policy

MODEL_CHECKPOINT = "google/flan-t5-base"  # other options: t5-base, facebook/bart-base, gpt2, etc...
USE_TOKENIZER_LENGTH_POLICY = True

# Defaults if model is not listed in overrides.
DEFAULT_MAX_INPUT_TOKENS = 512
DEFAULT_MIN_TARGET_TOKENS = 3
DEFAULT_MAX_TARGET_TOKENS = 32

# Only drop very extreme long inputs (after this value)
# Keep it larger than MAX_INPUT_TOKENS because normal long rows will be truncated, not dropped
HARD_DROP_INPUT_TOKENS = 1400

# Optional per-model overrides
MODEL_TOKEN_LIMITS = {
    "google/flan-t5-small": {"max_input": 512, "max_target": 32},
    "google/flan-t5-base":  {"max_input": 512, "max_target": 48},
    "t5-small":             {"max_input": 512, "max_target": 32},
    "t5-base":              {"max_input": 512, "max_target": 48},
    "facebook/bart-base":   {"max_input": 1024, "max_target": 64},
    #because GPT-2 has a total context limit of around 1024
    #so we should leave space for the generated headline
    #(960 input tokens + 64 generated tokens = 1024 total tokens)
    "gpt2":                 {"max_input": 960, "max_target": 64},
}

cfg = MODEL_TOKEN_LIMITS.get(MODEL_CHECKPOINT, {})
MAX_INPUT_TOKENS = cfg.get("max_input", DEFAULT_MAX_INPUT_TOKENS)
MIN_TARGET_TOKENS = DEFAULT_MIN_TARGET_TOKENS
MAX_TARGET_TOKENS = cfg.get("max_target", DEFAULT_MAX_TARGET_TOKENS)

if USE_TOKENIZER_LENGTH_POLICY and HAS_TRANSFORMERS:
    TOKENIZER_NAME = MODEL_CHECKPOINT
    print(f"Applying tokenizer-length policy with: {TOKENIZER_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)

    # Avoiding noisy warnings while we inspect long sequence lengths
    tokenizer.model_max_length = 10_000

    # Count raw token lengths first
    model_df["input_token_count_raw"] = model_df["article_clean_transformer"].apply(
        lambda x: len(tokenizer.encode(str(x), add_special_tokens=True))
    )
    model_df["target_token_count"] = model_df["title_clean"].apply(
        lambda x: len(tokenizer.encode(str(x), add_special_tokens=True))
    )

    # Mark rows that exceed model input limit
    # NOT mutating article_clean_transformer here
    # keeping full cleaned text and apply truncation later per model alias
    model_df["was_input_truncated"] = model_df["input_token_count_raw"] > MAX_INPUT_TOKENS
    trunc_count = int(model_df["was_input_truncated"].sum())
    print(f"Rows above {MAX_INPUT_TOKENS} input tokens (will be truncated later per model): {trunc_count}")

    # For observing: capped token count as if truncated at MAX_INPUT_TOKENS
    model_df["input_token_count"] = model_df["input_token_count_raw"].clip(upper=MAX_INPUT_TOKENS)

    # Final token-level filtering:
    # dropping only extreme input outliers
    # enforcing target token bounds
    before_tok_df = model_df.copy()
    model_df = model_df[
        (model_df["input_token_count_raw"] <= HARD_DROP_INPUT_TOKENS) &
        (model_df["target_token_count"] >= MIN_TARGET_TOKENS) &
        (model_df["target_token_count"] <= MAX_TARGET_TOKENS)
    ].copy()

    log_filter_step(
        step_name="Tokenizer policy (truncate long, drop extreme outliers)",
        reason=(
            "Word count is approximate; token counts match transformer behavior. "
            "We keep more data by truncating normal long inputs and dropping only extreme outliers."
        ),
        before_df=before_tok_df,
        after_df=model_df,
        show_cols=[
            "title_clean",
            "input_token_count_raw",
            "input_token_count",
            "target_token_count",
            "was_input_truncated"
        ]
    )

else:
    log_skipped_step(
        step_name="Tokenizer policy (truncate long, drop extreme outliers)",
        reason="Skipped because transformers is missing or USE_TOKENIZER_LENGTH_POLICY=False.",
        current_df=model_df
    )

print("Word-count stats after filtering:")
display(model_df[["input_word_count", "target_word_count"]].describe())

if "input_token_count_raw" in model_df.columns:
    print("Raw input token stats (before truncation):")
    display(model_df["input_token_count_raw"].describe())

if "input_token_count" in model_df.columns:
    print("Final input token stats (after truncation):")
    display(model_df["input_token_count"].describe())

Step: Word-length filter
Why this step: Keep rows where article has enough context and headline length looks realistic for training.
Rows before: 49728
Rows after : 48240
Removed    : 1488 (2.992% of this step input)
Cumulative removed from start: 1620 (3.249%)
Sample removed rows (up to 5):


,title_clean,input_word_count,target_word_count,article_clean_transformer,article_preview,title_preview
81,Facebook to launch hand tracking feature on Oculus Quest,50,9,"Sept 25 (Reuters) - Facebook Inc will bring a hand tracking feature to its virtual reality headset Oculus Quest, Chief Executive Officer Mark Zuckerberg sai...","Sept 25 (Reuters) - Facebook Inc will bring a hand tracking feature to its virtual reality headset Oculus Quest, Chief Executive Officer Mark Zuckerberg sai...",Facebook to launch hand tracking feature on Oculus Quest
147,RHOD Star D'Andra Simmons Reveals the 'Shake Up' She's Facing As She Takes Over the Family Business,59,17,The Real Housewives Of Dallas premieres August 15th At 9PM ET. Come back every weekday at 12:00 p.m. EST to watch People Now streaming live from the Meredit...,The Real Housewives Of Dallas premieres August 15th At 9PM ET. Come back every weekday at 12:00 p.m. EST to watch People Now streaming live from the Meredit...,RHOD Star D'Andra Simmons Reveals the 'Shake Up' She's Facing As She Takes Over the Family Business
179,U.S. Senate Republican leader blasts House's coronavirus bill,78,8,"WASHINGTON, March 12 (Reuters) - A wide-ranging coronavirus response bill sought by Democrats in the U.S. House of Representatives was criticized on Thursda...","WASHINGTON, March 12 (Reuters) - A wide-ranging coronavirus response bill sought by Democrats in the U.S. House of Representatives was criticized on Thursda...",U.S. Senate Republican leader blasts House's coronavirus bill
247,U.S. House Speaker Pelosi requests intelligence briefings on Saudi attacks,68,10,"WASHINGTON, Sept 17 (Reuters) - U.S. House of Representatives Speaker Nancy Pelosi has requested a briefing for all House members on the Saudi Aramco attack...","WASHINGTON, Sept 17 (Reuters) - U.S. House of Representatives Speaker Nancy Pelosi has requested a briefing for all House members on the Saudi Aramco attack...",U.S. House Speaker Pelosi requests intelligence briefings on Saudi attacks
256,Saudi's Jarir Marketing proposes lower Q2 dividend,66,7,"DUBAI, July 24 (Reuters) - The board of Jarir Marketing , one of Saudi Arabia's largest retailers by market value, has proposed paying a cash dividend of 1....","DUBAI, July 24 (Reuters) - The board of Jarir Marketing , one of Saudi Arabia's largest retailers by market value, has proposed paying a cash dividend of 1....",Saudi's Jarir Marketing proposes lower Q2 dividend


Applying tokenizer-length policy with: google/flan-t5-base
Rows above 512 input tokens (will be truncated later per model): 26280
Step: Tokenizer policy (truncate long, drop extreme outliers)
Why this step: Word count is approximate; token counts match transformer behavior. We keep more data by truncating normal long inputs and dropping only extreme outliers.
Rows before: 48240
Rows after : 47677
Removed    : 563 (1.167% of this step input)
Cumulative removed from start: 2183 (4.378%)
Sample removed rows (up to 5):


,title_clean,input_token_count_raw,input_token_count,target_token_count,was_input_truncated,article_clean_transformer,article_preview,title_preview
97,Nearly half of voters say they're unsure who won Detroit debates | TheHill,1406,512,21,True,"Almost half of voters polled - 47 percent - say there was no clear winner from last week's Democratic presidential primary debate in Detroit, according to a...","Almost half of voters polled - 47 percent - say there was no clear winner from last week's Democratic presidential primary debate in Detroit, according to a...",Nearly half of voters say they're unsure who won Detroit debates | TheHill
268,"Review: The 2019 GMC Sierra AT4 is the off-road truck, refined",1438,512,18,True,"Since the launch of the Ford F-150 Raptor, we've been waiting for General Motors to hit back. When we heard about the GMC Sierra AT4, an off-road focused tr...","Since the launch of the Ford F-150 Raptor, we've been waiting for General Motors to hit back. When we heard about the GMC Sierra AT4, an off-road focused tr...","Review: The 2019 GMC Sierra AT4 is the off-road truck, refined"
611,Top Iranian general Qasem Soleimani killed in US airstrike in Baghdad: Pentagon,1441,512,26,True,"WASHINGTON - Iran's top commander, Gen. Qasem Soleimani, has been killed in a U.S. airstrike in Baghdad, the Pentagon confirmed Thursday night following rep...","WASHINGTON - Iran's top commander, Gen. Qasem Soleimani, has been killed in a U.S. airstrike in Baghdad, the Pentagon confirmed Thursday night following rep...",Top Iranian general Qasem Soleimani killed in US airstrike in Baghdad: Pentagon
671,"The Baltics fear European ""strategic autonomy"" - Charlemagne: In Europe's McCainland",1402,512,27,True,"PERHAPS nowhere in Europe was John McCain mourned more deeply than in Estonia, Latvia and Lithuania. He had been one of a small group of American senators w...","PERHAPS nowhere in Europe was John McCain mourned more deeply than in Estonia, Latvia and Lithuania. He had been one of a small group of American senators w...","The Baltics fear European ""strategic autonomy"" - Charlemagne: In Europe's McCainland"
713,7 Things to Do With Your Kids in N.Y.C. This Weekend,1567,512,17,True,Our guide to cultural events in New York City for children and teenagers happening this weekend and in the week ahead. ARTSEE: KIKI SMITH at the Museum at E...,Our guide to cultural events in New York City for children and teenagers happening this weekend and in the week ahead. ARTSEE: KIKI SMITH at the Museum at E...,7 Things to Do With Your Kids in N.Y.C. This Weekend


Word-count stats after filtering:


,input_word_count,target_word_count
count,47677.000000,47677.000000
mean,417.618390,10.148625
std,229.157581,2.577284
min,80.000000,3.000000
25%,236.000000,8.000000
50%,380.000000,10.000000
75%,564.000000,12.000000
max,1011.000000,22.000000


Raw input token stats (before truncation):


,input_token_count_raw
count,47677.000000
mean,595.303249
std,321.812863
min,94.000000
25%,338.000000
50%,543.000000
75%,806.000000
max,1400.000000


Final input token stats (after truncation):


,input_token_count
count,47677.000000
mean,422.465948
std,124.724631
min,94.000000
25%,338.000000
50%,512.000000
75%,512.000000
max,512.000000


## 13. Headline Quality Filters

Some titles are still noisy even after basic cleaning.

Here applying simple quality heuristics to remove targets that are usually bad labels for training:

- too much punctuation/symbols
- too many digits
- all-caps shouting titles
- generic placeholders like "video", "live updates", etc.

In [17]:
GENERIC_BAD_TITLES = {
    "video", "photos", "live updates", "live blog", "breaking", "update", "brief"
}

def title_quality_metrics(title):
    t = str(title).strip()
    if not t:
        return pd.Series({
            "title_is_bad_generic": True,
            "title_punct_ratio": 1.0,
            "title_digit_ratio": 1.0,
            "title_upper_ratio": 1.0
        })

    letters = [c for c in t if c.isalpha()]
    digits = [c for c in t if c.isdigit()]
    punct = [c for c in t if not c.isalnum() and not c.isspace()]

    letter_count = max(len(letters), 1)
    char_count = max(len(t.replace(" ", "")), 1)

    upper_ratio = sum(1 for c in letters if c.isupper()) / letter_count
    digit_ratio = len(digits) / char_count
    punct_ratio = len(punct) / char_count

    low = t.lower()
    is_bad_generic = low in GENERIC_BAD_TITLES

    return pd.Series({
        "title_is_bad_generic": is_bad_generic,
        "title_punct_ratio": punct_ratio,
        "title_digit_ratio": digit_ratio,
        "title_upper_ratio": upper_ratio
    })

quality_df = model_df["title_clean"].apply(title_quality_metrics)
model_df = pd.concat([model_df, quality_df], axis=1)

before_df = model_df.copy()
model_df = model_df[
    (~model_df["title_is_bad_generic"]) &
    (model_df["title_punct_ratio"] <= 0.35) &
    (model_df["title_digit_ratio"] <= 0.40) &
    (model_df["title_upper_ratio"] <= 0.90)
].copy()

log_filter_step(
    step_name="Headline quality filter",
    reason="Remove headline targets that look noisy or low-information (generic labels, punctuation-heavy, etc.).",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "title_is_bad_generic", "title_punct_ratio", "title_digit_ratio", "title_upper_ratio"]
)

Step: Headline quality filter
Why this step: Remove headline targets that look noisy or low-information (generic labels, punctuation-heavy, etc.).
Rows before: 47677
Rows after : 47659
Removed    : 18 (0.038% of this step input)
Cumulative removed from start: 2201 (4.414%)
Sample removed rows (up to 5):


,title_clean,title_is_bad_generic,title_punct_ratio,title_digit_ratio,title_upper_ratio,article_clean_transformer,article_preview,title_preview
10081,"CNN 10 - April 24, 2019",False,0.111111,0.444444,0.5,"April 24, 2019 We're featuring a variety of stories from around the world this Wednesday, from an analysis of Boeing's struggles to a preview of an upcoming...","April 24, 2019 We're featuring a variety of stories from around the world this Wednesday, from an analysis of Boeing's struggles to a preview of an upcoming...","CNN 10 - April 24, 2019"
14496,"CNN 10 - April 16, 2018",False,0.111111,0.444444,0.5,"April 16, 2018 The U.S., the U.K. and France have launched airstrikes on three targets in Syria, and today's show explains why and brings you some of the in...","April 16, 2018 The U.S., the U.K. and France have launched airstrikes on three targets in Syria, and today's show explains why and brings you some of the in...","CNN 10 - April 16, 2018"
14520,IFR US ECM WEEKLY WRAP-UP,False,0.047619,0.000000,1.0,"NEW YORK, Jan 24 (IFR) - WEEKLY TOTAL $4.58bn - IPO $800m - ABB/BLOCK $782m - FOLLOW-ON $2.45bn - CB $550m TUESDAY Adaptimmune Therapeutics (UK, biotech) - ...","NEW YORK, Jan 24 (IFR) - WEEKLY TOTAL $4.58bn - IPO $800m - ABB/BLOCK $782m - FOLLOW-ON $2.45bn - CB $550m TUESDAY Adaptimmune Therapeutics (UK, biotech) - ...",IFR US ECM WEEKLY WRAP-UP
16261,"CNN 10 - April 9, 2018",False,0.117647,0.411765,0.5,"April 9, 2018 CNN 10 starts a new week by explaining events concerning the U.S. National Guard, politics in Hungary, and a massive crack in the earth in Ken...","April 9, 2018 CNN 10 starts a new week by explaining events concerning the U.S. National Guard, politics in Hungary, and a massive crack in the earth in Ken...","CNN 10 - April 9, 2018"
17046,"CNN 10 - March 5, 2019",False,0.117647,0.411765,0.5,"March 5, 2019 A natural disaster in the U.S. state of Alabama leads off our show. And after reporting on the destructive effects of several tornadoes, we'll...","March 5, 2019 A natural disaster in the U.S. state of Alabama leads off our show. And after reporting on the destructive effects of several tornadoes, we'll...","CNN 10 - March 5, 2019"


### 13.1 Clickbait-Style Headline Filter

This removes titles that are technically clean but not suitable for professional news headline generation.

In [18]:
CLICKBAIT_PATTERNS = [
    r"you won.?t believe",
    r"what happened next",
    r"this is why",
    r"everyone is talking",
    r"will shock you",
    r"the internet is going crazy",
    r"can.?t stop talking",
    r"things you need to know",
    r"here.?s why",
    r"here is why"
]

combined_clickbait_pattern = re.compile("|".join(CLICKBAIT_PATTERNS), flags=re.IGNORECASE)

before_df = model_df.copy()

model_df = model_df[
    ~model_df["title_clean"].str.contains(combined_clickbait_pattern, na=False, regex=True)
].copy()

log_filter_step(
    step_name="Clickbait-style headline filter",
    reason="Remove overly clickbait-style headlines that may weaken professional headline generation.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean"]
)

Step: Clickbait-style headline filter
Why this step: Remove overly clickbait-style headlines that may weaken professional headline generation.
Rows before: 47659
Rows after : 47605
Removed    : 54 (0.113% of this step input)
Cumulative removed from start: 2255 (4.523%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
1817,Coronavirus: Here's why investors shouldn't compare the outbreak to SARS,"As tempting as it may be to compare the coronavirus outbreak to the SARS epidemic from 2002 to 2003, a number of Wall Street analysts warn this is a dangero...","As tempting as it may be to compare the coronavirus outbreak to the SARS epidemic from 2002 to 2003, a number of Wall Street analysts warn this is a dangero...",Coronavirus: Here's why investors shouldn't compare the outbreak to SARS
4187,"We're nearing the 100-day mark, and Donald Trump can't stop talking about the election.","Since Trump was inaugurated in January, millions of people have protested his racist, misogynistic rhetoric and his racist, misogynistic administration. His...","Since Trump was inaugurated in January, millions of people have protested his racist, misogynistic rhetoric and his racist, misogynistic administration. His...","We're nearing the 100-day mark, and Donald Trump can't stop talking about the election."
4456,Here's why retailers should be scared of Amazon dominating e-commerce,There's new and staggering data that support the fact that retailers should increasingly be scared of Amazon dominating e-commerce. When consumers are ready...,There's new and staggering data that support the fact that retailers should increasingly be scared of Amazon dominating e-commerce. When consumers are ready...,Here's why retailers should be scared of Amazon dominating e-commerce
5481,8 Things You Need To Know This AM,"1. Major News: The FBI is not recommending criminal charges in the case involving Hillary Clinton's use of private email servers. ""We are expressing to [the...","1. Major News: The FBI is not recommending criminal charges in the case involving Hillary Clinton's use of private email servers. ""We are expressing to [the...",8 Things You Need To Know This AM
6083,Zika Is No Reason to Cancel the Olympics. Here's Why,"On June 14, the World Health Organization reiterated that the Zika outbreak did not pose enough riskto warrant moving, postponing, or cancelling the summer ...","On June 14, the World Health Organization reiterated that the Zika outbreak did not pose enough riskto warrant moving, postponing, or cancelling the summer ...",Zika Is No Reason to Cancel the Olympics. Here's Why


## 14. Optional Language Filter (English)

All The News is mostly English, but small language noise can still appear.

- keeping rows that look like English with good confidence

In [19]:
USE_LANGUAGE_FILTER = True
# Low threshold to reduce false removals of valid English news rows
MIN_ENGLISH_PROB = 0.80

def english_probability(text):
    text = str(text).strip()
    if not text:
        return 0.0
    if not HAS_LANGDETECT:
        return np.nan
    try:
        preds = detect_langs(text)
        for p in preds:
            if str(p.lang).lower() == "en":
                return float(p.prob)
        return 0.0
    except Exception:
        return 0.0

if USE_LANGUAGE_FILTER and HAS_LANGDETECT:
    # Add english_prob to model_df first
    # Use more context (up to 1200 chars) for more stable language detection
    check_text = model_df["title_clean"] + " " + model_df["article_clean_transformer"].str[:1200]
    model_df["english_prob"] = check_text.apply(english_probability)

    # Then capture before_df *after* english_prob has been added
    before_df = model_df.copy()

    # Then apply the filter
    model_df = model_df[model_df["english_prob"] >= MIN_ENGLISH_PROB].copy()

    log_filter_step(
        step_name="English language filter",
        reason="Remove probable non-English rows to keep target language consistent for generation.",
        before_df=before_df,
        after_df=model_df,
        show_cols=["title_clean", "english_prob"]
    )
else:
    log_skipped_step(
        step_name="English language filter",
        reason="Skipped because langdetect is missing or USE_LANGUAGE_FILTER=False.",
        current_df=model_df
    )

Step: English language filter
Why this step: Remove probable non-English rows to keep target language consistent for generation.
Rows before: 47605
Rows after : 47596
Removed    : 9 (0.019% of this step input)
Cumulative removed from start: 2264 (4.541%)
Sample removed rows (up to 5):


,title_clean,english_prob,article_clean_transformer,article_preview,title_preview
4778,PUNTO 1-Enel presenterà a breve un'offerta per Metroweb - fonti,0.000000,"(Aggiunge dettagli) By Silvia Aloisi and Stephen Jewkes ROMA, 5 maggio (Reuters) - Enel si appresta a presentare a breve un'offerta per l'operatore di fibra...","(Aggiunge dettagli) By Silvia Aloisi and Stephen Jewkes ROMA, 5 maggio (Reuters) - Enel si appresta a presentare a breve un'offerta per l'operatore di fibra...",PUNTO 1-Enel presenterà a breve un'offerta per Metroweb - fonti
9104,"Now you can read 'Don Quixote' in 17,000 tweets",0.428571,"Reading long classic books can be a frustrating experience for many, due to the daily distractions of contemporary life. That's why a retired Spanish engine...","Reading long classic books can be a frustrating experience for many, due to the daily distractions of contemporary life. That's why a retired Spanish engine...","Now you can read 'Don Quixote' in 17,000 tweets"
10104,"PUNTO 3-UniCredit lancia bond senior giugno 2025 da 1,25",0.000000,"(Aggiorna con importo, guidance finale) LONDRA, 18 giugno (Reuters) - UniCredit ha lanciato una nuova obbligazione senior dell'importo di 1,25 miliardi a sc...","(Aggiorna con importo, guidance finale) LONDRA, 18 giugno (Reuters) - UniCredit ha lanciato una nuova obbligazione senior dell'importo di 1,25 miliardi a sc...","PUNTO 3-UniCredit lancia bond senior giugno 2025 da 1,25"
12720,Actores gay brillan con varias victorias en los Emmy,0.000000,"LOS ANGELES, 23 sep (Reuters) - Intérpretes homosexuales brillaron el domingo en los premios Emmy con victorias en varias categorías, entre ellas la de mejo...","LOS ANGELES, 23 sep (Reuters) - Intérpretes homosexuales brillaron el domingo en los premios Emmy con victorias en varias categorías, entre ellas la de mejo...",Actores gay brillan con varias victorias en los Emmy
15557,"BOLSAS EUROPA-Tecnológicas, receios comércio pesam nas acções europeias",0.000000,3 Abr (Reuters) - As acções europeias seguem em queda esta terça-feira com os investidores a iniciarem o segundo trimestre numa atmosfera febril de tensões ...,3 Abr (Reuters) - As acções europeias seguem em queda esta terça-feira com os investidores a iniciarem o segundo trimestre numa atmosfera febril de tensões ...,"BOLSAS EUROPA-Tecnológicas, receios comércio pesam nas acções europeias"


## 15. Title-Article Alignment Check
This computes lexical overlap between title and article body.

Because abstractive headlines can have low lexical overlap, the filtering step is optional and disabled by default

In [20]:
def content_tokens(text):
    toks = re.findall(r"[A-Za-z]+", str(text).lower())
    return [t for t in toks if t not in ENGLISH_STOP_WORDS and len(t) > 2]

def title_article_overlap(title, article):
    tset = set(content_tokens(title))
    aset = set(content_tokens(article))
    if not tset:
        return 0.0
    return len(tset.intersection(aset)) / len(tset)

model_df["title_article_overlap"] = model_df.apply(
    lambda r: title_article_overlap(r["title_clean"], r["article_clean_transformer"]),
    axis=1
)

# This heuristic can remove valid abstractive headlines
# Keep it optional and using a softer threshold when enabled
USE_TITLE_ARTICLE_OVERLAP_FILTER = False
MIN_TITLE_ARTICLE_OVERLAP = 0.05

if USE_TITLE_ARTICLE_OVERLAP_FILTER:
    before_df = model_df.copy()
    model_df = model_df[model_df["title_article_overlap"] >= MIN_TITLE_ARTICLE_OVERLAP].copy()

    log_filter_step(
        step_name="Title-article overlap filter",
        reason="Optional weak-pair filter using lexical overlap (kept soft to avoid dropping abstractive headlines).",
        before_df=before_df,
        after_df=model_df,
        show_cols=["title_clean", "title_article_overlap"]
    )
else:
    log_skipped_step(
        step_name="Title-article overlap filter",
        reason="Skipped by default to preserve valid abstractive/acronym headlines.",
        current_df=model_df
    )

print("Overlap stats on current data:")
display(model_df["title_article_overlap"].describe())

Step: Title-article overlap filter
Why this step: Skipped by default to preserve valid abstractive/acronym headlines.
Step status: skipped (no rows removed).
Overlap stats on current data:


,title_article_overlap
count,47596.000000
mean,0.733049
std,0.184180
min,0.000000
25%,0.625000
50%,0.750000
75%,0.857143
max,1.000000


## 16. Optional Extra Noise Checks for Model Data

This section is still model-data preprocessing, not linguistic analysis.

It checks for remaining suspicious titles that may not be useful as headline targets.

Inspect first, then decide whether to remove more patterns.

In [22]:
remaining_noisy_title_patterns = [
    r"^\s*brief\b",
    r"^\s*update\s*\d*\s*[-:]",
    r"^\s*refile\s*-?\s*update\s*\d*\s*[-:]",
    r"^\s*wrapup\s*\d*\s*[-:]",
    r"^\s*press digest\b",
    r"^\s*factors to watch\b",
    r"^\s*preview\b",
    r"^\s*team by team analysis\b",
    r"^\s*pro rata podcast\b",
    r"^\s*cctv script\b",
    r"^\s*us stocks snapshot\b",
    r"^\s*rising:\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}\s*\|\s*TheHill\s*$",
]

combined_remaining_noise_pattern = re.compile("|".join(remaining_noisy_title_patterns), flags=re.IGNORECASE)

suspicious_titles = model_df[model_df["title_clean"].str.contains(combined_remaining_noise_pattern, na=False, regex=True)]
print("Suspicious/noisy titles found:", len(suspicious_titles))
print("These are candidates for removal in the next optional step.")
display(suspicious_titles[["title_clean", "article_clean_transformer"]].head(20))

Suspicious/noisy titles found: 0
These are candidates for removal in the next optional step.


,title_clean,article_clean_transformer


In [23]:
print("Checking for 'UPDATE' and 'PREVIEW' in the flagged suspicious titles:")
mask = suspicious_titles['title_clean'].str.contains('UPDATE|PREVIEW', case=False, na=False)
display(suspicious_titles[mask][['title_clean', 'article_clean_transformer']].head(10))

Checking for 'UPDATE' and 'PREVIEW' in the flagged suspicious titles:


,title_clean,article_clean_transformer


### Remove Remaining Noisy Titles

True if the previous inspection shows real remaining noise

In [24]:
REMOVE_REMAINING_NOISY_TITLES = True

if REMOVE_REMAINING_NOISY_TITLES:
    before_df = model_df.copy()
    model_df = model_df[~model_df["title_clean"].str.contains(combined_remaining_noise_pattern, na=False, regex=True)].copy()

    log_filter_step(
        step_name="Optional noisy-title regex filter",
        reason="Remove repeated newswire-style title templates that do not behave like normal headlines.",
        before_df=before_df,
        after_df=model_df,
        show_cols=["title_clean"]
    )
else:
    log_skipped_step(
        step_name="Optional noisy-title regex filter",
        reason="Skipped because REMOVE_REMAINING_NOISY_TITLES=False.",
        current_df=model_df
    )

# Final preprocessing audit summary up to this point
audit_df = pd.DataFrame(preprocess_audit)
print("\n" + "#" * 90)
print("PREPROCESSING AUDIT SUMMARY")
print("#" * 90)
if len(audit_df) > 0:
    display(audit_df)
    final_rows_now = len(model_df)
    total_removed = INITIAL_ROWS - final_rows_now
    total_removed_pct = (total_removed / INITIAL_ROWS * 100) if INITIAL_ROWS else 0.0

    print(f"Initial rows after cleaning stage: {INITIAL_ROWS}")
    print(f"Rows remaining after all filters : {final_rows_now}")
    print(f"Total removed by filters         : {total_removed} ({total_removed_pct:.3f}%)")

    # Grand total from the raw loaded data (includes early dropna step)
    if "raw_df" in globals():
        grand_total_removed = len(raw_df) - final_rows_now
        grand_total_removed_pct = (grand_total_removed / len(raw_df) * 100) if len(raw_df) else 0.0
        print(f"Grand total removed from raw_df : {grand_total_removed} ({grand_total_removed_pct:.3f}%)")
else:
    print("No audit records found.")

Step: Optional noisy-title regex filter
Why this step: Remove repeated newswire-style title templates that do not behave like normal headlines.
Rows before: 47596
Rows after : 47596
Removed    : 0 (0.000% of this step input)
Cumulative removed from start: 2264 (4.541%)
No rows removed in this step.

##########################################################################################
PREPROCESSING AUDIT SUMMARY
##########################################################################################


,step,reason,before_rows,after_rows,removed_rows,removed_pct_step,cumulative_removed,cumulative_removed_pct
0,Safety char-length filter,Remove rows that become too short after cleaning; very short body/title pairs are usually weak supervision.,49860,49854,6,0.012,6,0.012
1,Exact duplicate removal,Remove repeated article pairs to reduce memorization and data leakage across splits.,49854,49840,14,0.028,20,0.040
2,Loose near-duplicate removal,Catch practical near-duplicates with tiny edits so the model sees more diverse examples.,49840,49833,7,0.014,27,0.054
3,Embedding-based near-duplicate removal,Skipped by default because it can be slow; set USE_EMBEDDING_NEAR_DUPLICATE_FILTER=True to enable it.,49833,49833,0,0.000,27,0.054
4,Embedding-based near-duplicate removal,Remove semantically similar duplicate stories that text-based dedup may miss.,49833,49728,105,0.211,132,0.265
5,Word-length filter,Keep rows where article has enough context and headline length looks realistic for training.,49728,48240,1488,2.992,1620,3.249
6,"Tokenizer policy (truncate long, drop extreme outliers)",Word count is approximate; token counts match transformer behavior. We keep more data by truncating normal long inputs and dropping only extreme outliers.,48240,47677,563,1.167,2183,4.378
7,Headline quality filter,"Remove headline targets that look noisy or low-information (generic labels, punctuation-heavy, etc.).",47677,47659,18,0.038,2201,4.414
8,Clickbait-style headline filter,Remove overly clickbait-style headlines that may weaken professional headline generation.,47659,47605,54,0.113,2255,4.523
9,English language filter,Remove probable non-English rows to keep target language consistent for generation.,47605,47596,9,0.019,2264,4.541


Initial rows after cleaning stage: 49860
Rows remaining after all filters : 47596
Total removed by filters         : 2264 (4.541%)
Grand total removed from raw_df : 2264 (4.541%)


### 16.1 Publication Balance Analysis

This checks if one publisher dominates the dataset.
A dominated dataset may make the model learn one publisher's headline style too strongly.

In [25]:
if "publication" in model_df.columns:
    print("Publication distribution:")
    publication_counts = model_df["publication"].fillna("Unknown").value_counts()
    display(publication_counts.head(20))

    print("Publication percentage:")
    display((publication_counts / len(model_df) * 100).round(2).head(20))
else:
    print("No publication column found.")

Publication distribution:


,count
publication,
Reuters,13227
CNBC,4616
The Hill,4419
The New York Times,3495
People,3207
CNN,2556
Mashable,2170
Refinery 29,1945
Vice,1789


Publication percentage:


,count
publication,
Reuters,27.79
CNBC,9.70
The Hill,9.28
The New York Times,7.34
People,6.74
CNN,5.37
Mashable,4.56
Refinery 29,4.09
Vice,3.76


### 16.2 Section / Category Balance Analysis

This checks whether the dataset is dominated by one topic.

In [26]:
if "section" in model_df.columns:
    print("Section distribution:")
    section_counts = model_df["section"].fillna("Unknown").value_counts()
    display(section_counts.head(30))

    print("Section percentage:")
    display((section_counts / len(model_df) * 100).round(2).head(30))
else:
    print("No section column found.")

Section distribution:


,count
section,
Unknown,17110
World News,2414
Business News,2368
politics,1038
us,798
Politics,767
Market News,704
Wires,699
Sports News,655


Section percentage:


,count
section,
Unknown,35.95
World News,5.07
Business News,4.98
politics,2.18
us,1.68
Politics,1.61
Market News,1.48
Wires,1.47
Sports News,1.38


## 17. Create Model-Specific Input/Target Columns (All Models)

Instead of selecting one model, this step creates model-ready columns for all configured models in one dataframe.

For each model alias, we create:

- `{alias}_article`
- `{alias}_title`

Formatting rules:

- seq2seq models (T5/FLAN/BART): `summarize: ...`
- causal models (GPT-2): `Article: ...\nHeadline:`

Optional model-specific token truncation is applied so each model column respects its own input limit.

In [27]:
# Build model-ready columns for all configured model families
# We keep one cleaned base article/title, then format them per model
MODEL_CONFIGS = {
    "flan_t5_small": {
        "checkpoint": "google/flan-t5-small",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "flan_t5_base": {
        "checkpoint": "google/flan-t5-base",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "t5_small": {
        "checkpoint": "t5-small",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "t5_base": {
        "checkpoint": "t5-base",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "bart_base": {
        "checkpoint": "facebook/bart-base",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "gpt2": {
        "checkpoint": "gpt2",
        "family": "causal",
        "prefix": None
    },
}

# If this exists, it means long texts were already truncated upstream
base_article_series = model_df["article_clean_transformer"].astype(str)
base_title_series = model_df["title_clean"].astype(str)

# Optional per-model truncation for each article column
APPLY_MODEL_SPECIFIC_TRUNCATION = True

def truncate_text_by_tokens(text, tokenizer, max_tokens):
    ids = tokenizer(
        str(text),
        add_special_tokens=True,
        truncation=True,
        max_length=max_tokens
    )["input_ids"]
    return tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

created_article_cols = []
created_title_cols = []
tokenizer_cache = {}

for alias, cfg_model in MODEL_CONFIGS.items():
    checkpoint = cfg_model["checkpoint"]
    family = cfg_model["family"]

    limits = MODEL_TOKEN_LIMITS.get(checkpoint, {})
    max_input_tokens = limits.get("max_input", DEFAULT_MAX_INPUT_TOKENS)

    article_series = base_article_series.copy()

    if APPLY_MODEL_SPECIFIC_TRUNCATION and HAS_TRANSFORMERS:
        if checkpoint not in tokenizer_cache:
            tokenizer_cache[checkpoint] = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
        tok = tokenizer_cache[checkpoint]
        article_series = article_series.apply(lambda x: truncate_text_by_tokens(x, tok, max_input_tokens))

    article_col = f"{alias}_article"
    title_col = f"{alias}_title"

    if family == "seq2seq":
        model_df[article_col] = cfg_model["prefix"] + article_series
    elif family == "causal":
        model_df[article_col] = "Article: " + article_series + "\nHeadline:"
    else:
        raise ValueError(f"Unsupported family: {family}")

    model_df[title_col] = base_title_series

    created_article_cols.append(article_col)
    created_title_cols.append(title_col)

paired_model_columns = [c for pair in zip(created_article_cols, created_title_cols) for c in pair]

print("Final model dataframe shape:", model_df.shape)
print("Created model-specific pair columns:")
print(paired_model_columns)
display(model_df[paired_model_columns[:4]].head(3))

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Final model dataframe shape: (47596, 39)
Created model-specific pair columns:
['flan_t5_small_article', 'flan_t5_small_title', 'flan_t5_base_article', 'flan_t5_base_title', 't5_small_article', 't5_small_title', 't5_base_article', 't5_base_title', 'bart_base_article', 'bart_base_title', 'gpt2_article', 'gpt2_title']


,flan_t5_small_article,flan_t5_small_title,flan_t5_base_article,flan_t5_base_title
0,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife
1,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan
2,"summarize: This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,"summarize: This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill


## 18. Final Sanity Checks

These checks make sure the final model columns are clean and usable.

In [28]:
model_pair_columns = [
    c for c in model_df.columns
    if any(c.endswith(suffix) for suffix in ["_article", "_title"])
]

article_cols = [c for c in model_pair_columns if c.endswith("_article")]
title_cols = [c for c in model_pair_columns if c.endswith("_title")]

for col in article_cols:
    print(f"Missing {col}:", model_df[col].isna().sum())
    print(f"Empty {col}:", (model_df[col].str.strip() == "").sum())

for col in title_cols:
    print(f"Missing {col}:", model_df[col].isna().sum())
    print(f"Empty {col}:", (model_df[col].str.strip() == "").sum())

for article_col in article_cols:
    base = article_col.replace("_article", "")
    title_col = f"{base}_title"
    if title_col in model_df.columns:
        dup_pairs = model_df[[article_col, title_col]].duplicated().sum()
        print(f"Duplicate pairs for {base}:", dup_pairs)

print("Final shape:", model_df.shape)

Missing flan_t5_small_article: 0
Empty flan_t5_small_article: 0
Missing flan_t5_base_article: 0
Empty flan_t5_base_article: 0
Missing t5_small_article: 0
Empty t5_small_article: 0
Missing t5_base_article: 0
Empty t5_base_article: 0
Missing bart_base_article: 0
Empty bart_base_article: 0
Missing gpt2_article: 0
Empty gpt2_article: 0
Missing flan_t5_small_title: 0
Empty flan_t5_small_title: 0
Missing flan_t5_base_title: 0
Empty flan_t5_base_title: 0
Missing t5_small_title: 0
Empty t5_small_title: 0
Missing t5_base_title: 0
Empty t5_base_title: 0
Missing bart_base_title: 0
Empty bart_base_title: 0
Missing gpt2_title: 0
Empty gpt2_title: 0
Duplicate pairs for flan_t5_small: 0
Duplicate pairs for flan_t5_base: 0
Duplicate pairs for t5_small: 0
Duplicate pairs for t5_base: 0
Duplicate pairs for bart_base: 0
Duplicate pairs for gpt2: 0
Final shape: (47596, 39)


## 19. Keep Only Useful Columns for the Model Files

 we keep only model-specific article/title pairs

So the exported files contain only columns like:

- `flan_t5_base_article`, `flan_t5_base_title`
- `bart_base_article`, `bart_base_title`
- `gpt2_article`, `gpt2_title`

No metadata or analysis columns are kept in final model files

In [29]:
cols_to_save = [
    c for c in model_df.columns
    if any(c.endswith(suffix) for suffix in ["_article", "_title"])
]

model_ready_df = model_df[cols_to_save].copy()

print("Columns saved for model-ready data (article/title pairs only):")
print(model_ready_df.columns.tolist())
print("Shape:", model_ready_df.shape)
display(model_ready_df.head(3))

Columns saved for model-ready data (article/title pairs only):
['flan_t5_small_article', 'flan_t5_small_title', 'flan_t5_base_article', 'flan_t5_base_title', 't5_small_article', 't5_small_title', 't5_base_article', 't5_base_title', 'bart_base_article', 'bart_base_title', 'gpt2_article', 'gpt2_title']
Shape: (47596, 12)


,flan_t5_small_article,flan_t5_small_title,flan_t5_base_article,flan_t5_base_title,t5_small_article,t5_small_title,t5_base_article,t5_base_title,bart_base_article,bart_base_title,gpt2_article,gpt2_title
0,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"summarize: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Jan...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife,"Article: Samsung's Galaxy S8 blends beautiful design and performance effortlessly, but how much safer is its battery compared to the Galaxy Note 7? In Janua...",It sure looks like the Galaxy S8's battery won't explode even if you stab it with a knife
1,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan,"summarize: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer...",Five Centuries of Drawings at the Morgan,"Article: The almost unbearably excellent show ""Drawn to Greatness: Master Drawings from the Thaw Collection"" begins with a love story. In 1954, the dealer E...",Five Centuries of Drawings at the Morgan
2,"summarize: This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,"summarize: This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,"summarize: This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,"summarize: This week, Rep. Joaquin CastroJoaquin CastroThe exhaustion of Democrats' anti-Trump delusions Juan Williams: Democrats finally hit Trump where it...",Retaliation tweets like Castro's to discourage political involvement must stop | TheHill,"summarize: This week, Rep. Joaquin CastroJoa

## 20. Train / Validation / Test Split

 split:

- 80% training
- 10% validation
- 10% testing

Split strategy is configurable:

- `temporal` (default): oldest news in train, newest in validation/test (more realistic)
- `group_random`: group-aware random split using `dedup_key_loose`
- `random`: standard random split fallback

## 21. Model-Ready Handoff Checks

This check makes sure the final dataset is immediately usable for training

I verify that:

- only model pair columns exist (`*_article`, `*_title`)
- every model alias has both columns
- there are no missing/empty values in train/validation/test

In [30]:
def validate_model_ready_split(df, split_name):
    pair_cols = [c for c in df.columns if c.endswith("_article") or c.endswith("_title")]
    non_pair_cols = [c for c in df.columns if c not in pair_cols]

    if non_pair_cols:
        raise ValueError(f"{split_name}: found non-pair columns: {non_pair_cols}")

    aliases = sorted({c.rsplit("_", 1)[0] for c in pair_cols})
    missing_pairs = []
    for a in aliases:
        if f"{a}_article" not in df.columns or f"{a}_title" not in df.columns:
            missing_pairs.append(a)

    if missing_pairs:
        raise ValueError(f"{split_name}: missing pair columns for aliases: {missing_pairs}")

    for a in aliases:
        ac = f"{a}_article"
        tc = f"{a}_title"
        na_count = int(df[[ac, tc]].isna().any(axis=1).sum())
        empty_count = int(((df[ac].astype(str).str.strip() == "") | (df[tc].astype(str).str.strip() == "")).sum())
        if na_count > 0 or empty_count > 0:
            raise ValueError(
                f"{split_name}: alias {a} has {na_count} NA rows and {empty_count} empty rows"
            )

    print(f"{split_name}: ready ({len(aliases)} model aliases, {len(pair_cols)} columns, {len(df)} rows)")


SPLIT_STRATEGY = "group_random"  # options: "temporal", "group_random", "random"

if SPLIT_STRATEGY == "temporal":
    if "date" not in model_df.columns:
        raise ValueError("Temporal split requested but `date` column is missing in model_df.")

    split_dates = pd.to_datetime(model_df["date"], errors="coerce")
    valid_mask = split_dates.notna()

    if not valid_mask.all():
        print(f"Temporal split warning: {(~valid_mask).sum()} rows have invalid dates and will be placed at the end.")

    sort_keys = split_dates.fillna(pd.Timestamp.max)
    ordered_pos = np.argsort(sort_keys.values)

    n = len(ordered_pos)
    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_pos = ordered_pos[:train_end]
    val_pos = ordered_pos[train_end:val_end]
    test_pos = ordered_pos[val_end:]

    train_df = model_ready_df.iloc[train_pos].copy()
    val_df = model_ready_df.iloc[val_pos].copy()
    test_df = model_ready_df.iloc[test_pos].copy()

    print("Used temporal split (oldest->train, newest->validation/test).")

elif SPLIT_STRATEGY == "group_random" and "dedup_key_loose" in model_df.columns:
    # Use grouping from model_df to avoid leakage,
    # while train/val/test files keep only model pair columns.
    gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_pos, temp_pos = next(gss_1.split(model_df, groups=model_df["dedup_key_loose"]))

    train_df = model_ready_df.iloc[train_pos].copy()
    temp_ready_df = model_ready_df.iloc[temp_pos].copy()
    temp_groups = model_df.iloc[temp_pos]["dedup_key_loose"].reset_index(drop=True)

    gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    val_rel_pos, test_rel_pos = next(gss_2.split(temp_ready_df, groups=temp_groups))

    val_df = temp_ready_df.iloc[val_rel_pos].copy()
    test_df = temp_ready_df.iloc[test_rel_pos].copy()

    print("Used group-aware random split with dedup_key_loose.")

else:
    train_df, temp_df = train_test_split(
        model_ready_df,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    print("Used normal random split.")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("Split percentages:")
print("Train %:", round(len(train_df) / len(model_ready_df) * 100, 2))
print("Validation %:", round(len(val_df) / len(model_ready_df) * 100, 2))
print("Test %:", round(len(test_df) / len(model_ready_df) * 100, 2))

validate_model_ready_split(model_ready_df, "full")
validate_model_ready_split(train_df, "train")
validate_model_ready_split(val_df, "validation")
validate_model_ready_split(test_df, "test")

Used group-aware random split with dedup_key_loose.
Train: (38076, 12)
Validation: (4760, 12)
Test: (4760, 12)
Split percentages:
Train %: 80.0
Validation %: 10.0
Test %: 10.0
full: ready (6 model aliases, 12 columns, 47596 rows)
train: ready (6 model aliases, 12 columns, 38076 rows)
validation: ready (6 model aliases, 12 columns, 4760 rows)
test: ready (6 model aliases, 12 columns, 4760 rows)


In [31]:
SPLIT_STRATEGY = "group_random"  # options: "temporal", "group_random", "random"

if SPLIT_STRATEGY == "temporal":
    if "date" not in model_df.columns:
        raise ValueError("Temporal split requested but `date` column is missing in model_df.")

    split_dates = pd.to_datetime(model_df["date"], errors="coerce")
    valid_mask = split_dates.notna()

    if not valid_mask.all():
        print(f"Temporal split warning: {(~valid_mask).sum()} rows have invalid dates and will be placed at the end.")

    sort_keys = split_dates.fillna(pd.Timestamp.max)
    ordered_pos = np.argsort(sort_keys.values)

    n = len(ordered_pos)
    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_pos = ordered_pos[:train_end]
    val_pos = ordered_pos[train_end:val_end]
    test_pos = ordered_pos[val_end:]

    train_df = model_ready_df.iloc[train_pos].copy()
    val_df = model_ready_df.iloc[val_pos].copy()
    test_df = model_ready_df.iloc[test_pos].copy()

    print("Used temporal split (oldest->train, newest->validation/test).")

elif SPLIT_STRATEGY == "group_random" and "dedup_key_loose" in model_df.columns:
    # Use grouping from model_df to avoid leakage,
    # while train/val/test files keep only model pair columns.
    gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_pos, temp_pos = next(gss_1.split(model_df, groups=model_df["dedup_key_loose"]))

    train_df = model_ready_df.iloc[train_pos].copy()
    temp_ready_df = model_ready_df.iloc[temp_pos].copy()
    temp_groups = model_df.iloc[temp_pos]["dedup_key_loose"].reset_index(drop=True)

    gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    val_rel_pos, test_rel_pos = next(gss_2.split(temp_ready_df, groups=temp_groups))

    val_df = temp_ready_df.iloc[val_rel_pos].copy()
    test_df = temp_ready_df.iloc[test_rel_pos].copy()

    print("Used group-aware random split with dedup_key_loose.")

else:
    train_df, temp_df = train_test_split(
        model_ready_df,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    print("Used normal random split.")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("Split percentages:")
print("Train %:", round(len(train_df) / len(model_ready_df) * 100, 2))
print("Validation %:", round(len(val_df) / len(model_ready_df) * 100, 2))
print("Test %:", round(len(test_df) / len(model_ready_df) * 100, 2))

Used group-aware random split with dedup_key_loose.
Train: (38076, 12)
Validation: (4760, 12)
Test: (4760, 12)
Split percentages:
Train %: 80.0
Validation %: 10.0
Test %: 10.0


## 22. Save Final Model Data Files

These files are ready for direct model training.

two formats are saved:

- wide files with all model pair columns (`*_article`, `*_title`)
- per-model files with standardized columns (`model_input`, `model_target`)

In [ ]:
train_path = os.path.join(OUTPUT_DIR, "news_headline_train_wide.csv")
val_path = os.path.join(OUTPUT_DIR, "news_headline_validation_wide.csv")
test_path = os.path.join(OUTPUT_DIR, "news_headline_test_wide.csv")
full_path = os.path.join(OUTPUT_DIR, "news_headline_model_ready_full_wide.pkl")

# 1) Save wide files (all model pairs in one table)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)
model_ready_df.to_pickle(full_path)

# 2) Save per-model files with standardized names for immediate training scripts
#    Each file has exactly two columns: model_input, model_target
aliases = sorted({c.rsplit("_", 1)[0] for c in model_ready_df.columns if c.endswith("_article")})
for alias in aliases:
    article_col = f"{alias}_article"
    title_col = f"{alias}_title"

    if article_col not in model_ready_df.columns or title_col not in model_ready_df.columns:
        continue

    alias_train = train_df[[article_col, title_col]].rename(columns={article_col: "model_input", title_col: "model_target"})
    alias_val = val_df[[article_col, title_col]].rename(columns={article_col: "model_input", title_col: "model_target"})
    alias_test = test_df[[article_col, title_col]].rename(columns={article_col: "model_input", title_col: "model_target"})

    alias_train.to_csv(os.path.join(OUTPUT_DIR, f"{alias}_train.csv"), index=False)
    alias_val.to_csv(os.path.join(OUTPUT_DIR, f"{alias}_validation.csv"), index=False)
    alias_test.to_csv(os.path.join(OUTPUT_DIR, f"{alias}_test.csv"), index=False)

print("Saved model-ready files:")
print(train_path)
print(val_path)
print(test_path)
print(full_path)
print("Per-model files saved for aliases:", aliases)